# GEO Photometry (star tracking, colour sensor)
Written by Kiyoaki Okudaira<br>
*Kyushu University Hanada Lab / University of Washington / IAU CPS SatHub<br>
(okudaira.kiyoaki.528@s.kyushu-u.ac.jp or kiyoaki@uw.edu)<br>
<br>
Light curves of a GEO object captured with a one-shot colour (Bayer) sensor under sidereal tracking:
one light curve per colour channel (R, G, B) and one for all colours together.<br>
<br>
**Method**<br>
- The Bayer phases are separated into sub-images (no interpolation): R, G1, G2, B at half resolution
  ("superpixel" grid); G = G1 + G2, ALL = sum of the four phases.<br>
- Field stars are fixed (sidereal tracking); the small residual drift is measured by phase correlation
  on the ALL image and all channels are registered with the same shift.<br>
- The median of the registered frames is the field-star image (the object moves and is rejected);
  it is subtracted per channel.<br>
- The object is detected automatically: residual blobs outside a star mask, fitted with a straight
  track by RANSAC using the expected GEO rate (about 15 arcsec/s) as a prior. The operator can review
  the track before the photometry (optional).<br>
- The registered ALL sum is plate-solved; Gaia DR3 (BP / G / RP) gives a zeropoint per channel.<br>
<br>
**History**<br>
coding 2026-10-22 : 1st coding<br>
<br>
(c) 2026 Kiyoaki Okudaira - Kyushu University Hanada Lab (SSDL) / University of Washington / IAU CPS SatHub

### Parameters
**Input files settings**

In [ ]:
import os

# Machine-specific roots (PATH_images, PATH_input, PATH_output, PATH_work)
# come from satphot_paths.py / paths.json.
from satphot_paths import *

# SER file of the colour capture (relative to PATH_images)
PATH_capture = PATH_images / "share2/KUPT/2023-11-01/20_02_29.ser"

# Frame range used for the light curve (1-based, inclusive; None = first / last frame)
frame_start = 1
frame_end = 235

**Run control**

In [ ]:
# "all", "1" (detection and review) or "2" (photometry; reloads the stage-1 state)
run_stage = os.environ.get("SATPHOT_STAGE", "all")

# Ask the operator to confirm the detected track before stage 2.
require_confirmation = os.environ.get("SATPHOT_REQUIRE_CONFIRMATION", "1") != "0"

# Approval flag (interactive: target_approved = True in the review cell;
# headless: SATPHOT_TARGET_APPROVED=1)
target_approved = os.environ.get("SATPHOT_TARGET_APPROVED", "0") == "1"

**Target and observation information**

In [ ]:
object_name = "EKRAN 2 DEB"
intl_des    = "1977-092H"
norad_id    = "11581"

observer_name = "Kyushu University Hanada Lab (SSDL)"
observatory   = "KUPT Kyushu University Pegasus Telescope"

**Colour sensor settings**

In [ ]:
# Bayer pattern: "auto" reads the SER ColorID (8 RGGB, 9 GRBG, 10 GBRG, 11 BGGR)
bayer_pattern = "auto"

# Channels measured. "ALL" is the sum of the four Bayer phases (2x2 superpixel),
# "G" is the mean of the two green phases.
channels = ["ALL", "R", "G", "B"]

# Gaia DR3 band used as the reference magnitude of each channel
reference_band = {"ALL": "phot_g_mean_mag", "R": "phot_rp_mean_mag", "G": "phot_g_mean_mag", "B": "phot_bp_mean_mag"}

# Pixel value regarded as saturated (16-bit sensor; applies to each Bayer phase)
saturation_level = 65000

**SER / camera settings**<br>Values found in a SharpCap CameraSettings.txt file next to the SER file override these.

In [ ]:
ser_exposure_seconds = 0.25
ser_gain = None
ser_offset = None
ser_camera_name = None

**Calibration settings**<br>Dark/flat (single FITS or a directory, same Bayer layout as the frames) are applied after the registration and astrometry; None = not used.

In [ ]:
PATH_dark = None
PATH_flat = None
PATH_flat_dark = None
calibration_scale = "auto"

**Registration and star subtraction**

In [ ]:
# Reference frame for the registration (1-based within the selected range; None = middle frame)
registration_reference_frame = None

# Star mask (used to ignore residuals of bright stars during the detection)
star_mask_sigma = 4.0
star_mask_dilation = 3

**Target detection settings**<br>Under sidereal tracking a GEO object moves at about 15 arcsec/s along a straight line.

In [ ]:
# Expected angular rate of the object [arcsec/s] and the accepted range (fraction)
expected_rate_arcsec_per_s = 15.04
rate_tolerance = (0.5, 1.5)

# Pixel scale of the superpixel grid [arcsec]; None = taken from the plate solve of a
# single frame in stage 1 (falls back to this value if given and the solve fails)
superpixel_scale_arcsec = None

# Residual-blob detection
detection_sigma = 4.0
detection_min_area = 3
detections_per_frame = 3

# RANSAC
ransac_iterations = 20000
ransac_inlier_distance = 2.0     # [superpixel]

# Manual track override: positions (superpixel coordinates of the registered grid)
# at two 1-based frame numbers; both None = automatic
manual_anchor_1 = None    # e.g. (1, 555.7, 290.0)
manual_anchor_2 = None    # e.g. (235, 155.1, 257.3)

**Measurement settings** (superpixel units)

In [ ]:
aperture_radius = 4.0
annulus_r_in    = 7.0
annulus_r_out   = 11.0

# Per-frame re-centring around the track prediction (on the star-subtracted ALL image)
recentre_half_size = 6
recentre_max_shift = 3.0

**Astrometry and zeropoint settings**

In [ ]:
astrometry_scale_low_arcsec_per_pixel = 1.5     # superpixel scale range
astrometry_scale_high_arcsec_per_pixel = 3.0
astrometry_center_hint = "auto"                 # "auto" (CameraSettings), (ra, dec) in deg, or None
astrometry_search_radius_deg = 5.0
astrometry_cpulimit_seconds = 600
astrometry_extra_options = {"--objs": 80, "--downsample": 2, "--no-remove-lines": True, "--uniformize": 0}

# Reference-star catalogue: "gaia" (Gaia DR3 through astroquery, needs network) or
# "index" (astrometry.net RDLS magnitudes, one band for every channel)
zeropoint_catalog = "gaia"
gaia_max_rows = 3000
reference_mag_min = 8.0
reference_mag_max = 15.0
zp_aperture_radius = 4.0
zp_annulus_r_in    = 7.0
zp_annulus_r_out   = 11.0
zp_centroid_box = 9
zp_max_centroid_shift = 2.5

**Software metadata**<br>Please DO NOT change from original.

In [ ]:
editor_app_name  = "SatPhotometry GEO StarTrack colour"
editor_version = "1.0.0"
creator_app_name = "SatPhotometry GEO StarTrack colour"
creator_version  = "1.0.0"

### Import and initial settings

In [ ]:
import json
import platform
import re
import shutil
import subprocess
import warnings
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

from astropy.io import fits
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.stats import SigmaClip, sigma_clipped_stats
from astropy.table import Table
from astropy.time import Time, TimeDelta
from astropy.visualization import ImageNormalize, ZScaleInterval, AsinhStretch
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.wcs.wcs import FITSFixedWarning
from photutils.aperture import ApertureStats, CircularAnnulus, CircularAperture, aperture_photometry
from photutils.centroids import centroid_2dg, centroid_sources
from tqdm.auto import tqdm
from IPython.display import Image, display

from satphotometry import serparser

warnings.simplefilter("ignore", FITSFixedWarning)

### General utilities

In [ ]:
def get_fits_files(directory):
    """Return non-hidden FITS files directly contained in a directory."""
    directory = Path(directory)
    if not directory.is_dir():
        raise NotADirectoryError(f"FITS directory does not exist: {directory}")

    files = [
        p for p in directory.iterdir()
        if p.is_file()
        and not p.name.startswith(".")
        and p.suffix.lower() in {".fit", ".fits", ".fts"}
    ]
    files.sort(key=lambda p: p.name.casefold())
    if not files:
        raise FileNotFoundError(f"No FITS files were found in: {directory}")
    return files


def recreate_directory(directory):
    """Delete and recreate a processing directory."""
    directory = Path(directory)
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)
    return directory


def get_observation_time(header):
    """Return exposure start, midpoint, and duration from a FITS header."""
    date_obs = header.get("DATE-OBS", header.get("DATEOBS"))
    if date_obs is None:
        raise KeyError("The FITS header contains neither DATE-OBS nor DATEOBS.")
    if "EXPTIME" not in header:
        raise KeyError("The FITS header does not contain EXPTIME.")

    exptime = float(header["EXPTIME"])
    if not np.isfinite(exptime) or exptime <= 0:
        raise ValueError(f"Invalid EXPTIME: {exptime}")

    try:
        time_start = Time(str(date_obs).strip(), format="fits", scale="utc")
    except ValueError:
        time_start = Time(str(date_obs).strip(), scale="utc")

    time_mid = time_start + TimeDelta(exptime / 2.0, format="sec")
    return time_start, time_mid, exptime


def windows_ticks_to_time(value):
    """Convert SER/Windows ticks since 0001-01-01 to Astropy Time."""
    unix_seconds = float(value) / 10_000_000.0 - 62_135_596_800.0
    return Time(unix_seconds, format="unix", scale="utc")


def parse_number(text):
    """Return the first floating-point number contained in text."""
    if text is None:
        return None
    match = re.search(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?", str(text))
    return None if match is None else float(match.group(0))


def parse_sharpcap_settings(path):
    """Read commonly used fields from a SharpCap CameraSettings.txt file."""
    path = Path(path)
    if not path.exists():
        return {}

    settings = {}
    with path.open("r", encoding="utf-8", errors="replace") as file:
        for line in file:
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            settings[key.strip()] = value.strip().strip('"')

    result = {}
    result["camera_name"] = settings.get("Camera") or settings.get("CameraName")
    result["camera_id"] = settings.get("CameraSerialNumber")

    binning = settings.get("Binning")
    if binning and "x" in binning.lower():
        x_bin, y_bin = re.split("x", binning, maxsplit=1, flags=re.IGNORECASE)
        result["x_binning"] = int(parse_number(x_bin))
        result["y_binning"] = int(parse_number(y_bin))

    exposure_text = settings.get("Exposure")
    exposure_value = parse_number(exposure_text)
    if exposure_value is not None:
        if "ms" in exposure_text.lower():
            exposure_value /= 1000.0
        elif "us" in exposure_text.lower() or "µs" in exposure_text.lower():
            exposure_value /= 1_000_000.0
        result["exposure"] = exposure_value

    key_map = {
        "Gain": "gain",
        "Offset": "offset",
        "Temperature": "ccd_temperature",
        "Target Temperature": "set_temperature",
    }
    for source_key, destination_key in key_map.items():
        value = parse_number(settings.get(source_key))
        if value is not None:
            result[destination_key] = value

    if "SharpCapVersion" in settings:
        result["swcreate"] = f"SharpCap {settings['SharpCapVersion']}"
    return result

### Calibration frames (from GEO_photometry_satTrack_integrated)

In [ ]:
def convert_ser_to_fits(
    ser_path,
    output_directory,
    *,
    start_frame=None,
    end_frame=None,
    exposure_seconds,
    gain=None,
    egain=None,
    egainsav=None,
    offset=None,
    x_binning=1,
    y_binning=1,
    camera_name=None,
    camera_id=None,
    ccd_temperature=None,
    set_temperature=None,
    observer=None,
    observatory=None,
    object_name=None,
    intl_des=None,
    norad_id=None,
):
    """Convert selected SER frames to FITS using satphotometry.serparser."""
    ser_path = Path(ser_path)
    output_directory = recreate_directory(output_directory)
    ser_file = serparser.Serfile(str(ser_path))
    frame_count = int(ser_file.getLength())

    first = 1 if start_frame is None else int(start_frame)
    last = frame_count if end_frame is None else int(end_frame)
    if first < 1 or last > frame_count or first > last:
        raise ValueError(
            f"Invalid SER frame range: {first}--{last}; total={frame_count}"
        )

    camera_settings_path = ser_path.with_suffix(".CameraSettings.txt")
    camera_settings = parse_sharpcap_settings(camera_settings_path)

    exposure_seconds = camera_settings.get("exposure", exposure_seconds)
    gain = camera_settings.get("gain", gain)
    offset = camera_settings.get("offset", offset)
    x_binning = camera_settings.get("x_binning", x_binning)
    y_binning = camera_settings.get("y_binning", y_binning)
    camera_name = camera_settings.get("camera_name", camera_name)
    camera_id = camera_settings.get("camera_id", camera_id)
    ccd_temperature = camera_settings.get("ccd_temperature", ccd_temperature)
    set_temperature = camera_settings.get("set_temperature", set_temperature)
    swcreate = camera_settings.get("swcreate", "satphotometry.serparser")

    try:
        timestamps = ser_file.readTrailFromHeader()
        if len(timestamps) != frame_count:
            timestamps = None
    except Exception:
        timestamps = None

    if timestamps is None:
        raise RuntimeError(
            "The SER file does not contain a usable per-frame timestamp trail. "
            "Reliable light-curve timing requires SER timestamps."
        )

    output_files = []
    stem = ser_path.stem
    for zero_index in tqdm(range(first - 1, last), desc="Converting SER to FITS"):
        output_path = output_directory / f"{stem}_{zero_index + 1:05d}.fits"
        ser_file.setCurrentPosition(zero_index)
        ser_file.saveFit(str(output_path))

        with fits.open(output_path, mode="update", memmap=False) as hdul:
            header = hdul[0].header
            time_start = windows_ticks_to_time(timestamps[zero_index])
            header["DATE-OBS"] = (time_start.fits, "Exposure start time UTC")
            header["EXPTIME"] = (float(exposure_seconds), "Exposure time [s]")
            header["XBINNING"] = int(x_binning)
            header["YBINNING"] = int(y_binning)
            header["SWCREATE"] = swcreate

            optional_header = {
                "GAIN": gain,
                "EGAIN": egain,
                "EGAINSAV": egainsav,
                "OFFSET": offset,
                "INSTRUME": camera_name,
                "CAMID": camera_id,
                "CCD-TEMP": ccd_temperature,
                "SET-TEMP": set_temperature,
                "OBSERVER": observer,
                "OBSERVAT": observatory,
                "OBJECT": object_name,
                "INTL-DES": intl_des,
                "NORAD-ID": norad_id,
            }
            for key, value in optional_header.items():
                if value is not None:
                    header[key] = value
            hdul.flush(output_verify="silentfix")

        output_files.append(output_path)
    return output_files


def create_master_dark(path):
    """Read a single dark frame or median-combine dark frames in a directory."""
    if path is None:
        return None
    path = Path(path)
    files = get_fits_files(path) if path.is_dir() else [path]
    stack = []
    for file_path in files:
        with fits.open(file_path, memmap=False) as hdul:
            stack.append(np.asarray(hdul[0].data, dtype=float))
    return np.nanmedian(np.stack(stack), axis=0)


def create_master_flat(path, master_flat_dark=None):
    """Read and normalize a single flat or a median-combined flat stack."""
    if path is None:
        return None
    path = Path(path)
    files = get_fits_files(path) if path.is_dir() else [path]
    stack = []
    for file_path in files:
        with fits.open(file_path, memmap=False) as hdul:
            data = np.asarray(hdul[0].data, dtype=float)
        if master_flat_dark is not None:
            data = data - master_flat_dark
        stack.append(data)
    master = np.nanmedian(np.stack(stack), axis=0)
    norm = np.nanmedian(master[np.isfinite(master)])
    if not np.isfinite(norm) or norm == 0:
        raise ValueError("The master flat cannot be normalized.")
    return master / norm


def calibrate_fits_files(fits_files, dark=None, flat=None):
    """Apply dark subtraction and flat-field correction in place."""
    if dark is None and flat is None:
        return
    for file_path in tqdm(fits_files, desc="Calibrating FITS frames"):
        with fits.open(file_path, mode="update", memmap=False) as hdul:
            data = np.asarray(hdul[0].data, dtype=float)
            if dark is not None:
                data = data - dark
            if flat is not None:
                with np.errstate(divide="ignore", invalid="ignore"):
                    data = data / flat
            hdul[0].data = data.astype(np.float32)
            hdul.flush(output_verify="silentfix")

### Aperture photometry (from GEO_photometry_satTrack_integrated)

In [ ]:
def detect_brightest_positive_source(
    image,
    detection_sigma=6.0,
    cutout_half_size=25,
):
    """Detect the brightest positive point-like residual in a 2-D image."""
    image = np.asarray(image, dtype=float)
    finite = np.isfinite(image)
    if not np.any(finite):
        raise ValueError("The detection image contains no finite pixels.")
    _, median, std = sigma_clipped_stats(image[finite], sigma=3.0, maxiters=5)
    if not np.isfinite(std) or std <= 0:
        raise ValueError("Could not estimate detection-image noise.")

    cleaned = np.where(finite, image, median)
    threshold = median + detection_sigma * std
    labels, number = ndimage.label(cleaned > threshold)
    if number == 0:
        raise RuntimeError("No target candidate was detected.")

    best_label = None
    best_flux = -np.inf
    for label_number in range(1, number + 1):
        region = labels == label_number
        flux = np.sum(np.clip(cleaned[region] - median, 0, None))
        if flux > best_flux:
            best_flux = flux
            best_label = label_number

    yy, xx = np.nonzero(labels == best_label)
    rough_x = float(np.mean(xx))
    rough_y = float(np.mean(yy))
    ny, nx = image.shape
    x0 = max(0, int(round(rough_x)) - cutout_half_size)
    x1 = min(nx, int(round(rough_x)) + cutout_half_size + 1)
    y0 = max(0, int(round(rough_y)) - cutout_half_size)
    y1 = min(ny, int(round(rough_y)) + cutout_half_size + 1)
    cutout = cleaned[y0:y1, x0:x1]
    _, local_median, _ = sigma_clipped_stats(cutout, sigma=3.0, maxiters=5)
    source = np.clip(cutout - local_median, 0, None)
    x_local, y_local = centroid_2dg(source)
    return float(x0 + x_local), float(y0 + y_local)


def build_target_detection_image(records, sample_count=30):
    """Median-combine positive star-subtracted samples for robust target detection."""
    if not records:
        raise ValueError("No star-subtracted records are available.")
    sample_indices = np.unique(
        np.linspace(0, len(records) - 1, min(sample_count, len(records))).astype(int)
    )
    stack = []
    for index in sample_indices:
        with fits.open(records[index]["photometry_path"], memmap=False) as hdul:
            stack.append(np.asarray(hdul[0].data, dtype=float))
    return np.nanmedian(np.stack(stack), axis=0)


def refine_centroid(image, x, y, half_size=8):
    """Refine a source centroid locally with a 2-D Gaussian centroid."""
    image = np.asarray(image, dtype=float)
    ny, nx = image.shape
    xc = int(round(float(x)))
    yc = int(round(float(y)))
    x0 = max(0, xc - half_size)
    x1 = min(nx, xc + half_size + 1)
    y0 = max(0, yc - half_size)
    y1 = min(ny, yc + half_size + 1)
    cutout = image[y0:y1, x0:x1]
    finite = np.isfinite(cutout)
    if np.count_nonzero(finite) < 9:
        raise ValueError("Too few finite pixels for centroid refinement.")
    _, median, _ = sigma_clipped_stats(cutout[finite], sigma=3.0, maxiters=5)
    source = np.where(finite, np.clip(cutout - median, 0, None), 0.0)
    x_local, y_local = centroid_2dg(source)
    return float(x0 + x_local), float(y0 + y_local)


def measure_object(image, x_object, y_object, aperture_radius, annulus_r_in, annulus_r_out):
    """Perform circular-aperture photometry with a sigma-clipped annulus."""
    image = np.asarray(image, dtype=float)
    position = (float(x_object), float(y_object))
    aperture = CircularAperture(position, r=float(aperture_radius))
    annulus = CircularAnnulus(
        position,
        r_in=float(annulus_r_in),
        r_out=float(annulus_r_out),
    )
    phot_table = aperture_photometry(image, aperture, method="exact")
    aperture_sum = float(phot_table["aperture_sum"][0])
    annulus_stats = ApertureStats(
        image,
        annulus,
        sigma_clip=SigmaClip(sigma=3.0, maxiters=5),
    )
    background_median = float(annulus_stats.median)
    aperture_area = float(aperture.area)
    background_sum = background_median * aperture_area
    object_flux = aperture_sum - background_sum
    return {
        "aperture_sum": aperture_sum,
        "background_median": background_median,
        "background_sum": background_sum,
        "aperture_area": aperture_area,
        "object_flux": object_flux,
    }


def read_photometric_zeropoint(magzero_path):
    """
    Prefer photometric_zeropoint_elec. If unavailable, use
    photometric_zeropoint_exp without detector-gain correction.
    """
    with Path(magzero_path).open("r", encoding="utf-8") as file:
        data = json.load(file)
    source = data.get("phot_params", data)

    zp_elec = source.get("photometric_zeropoint_elec", source.get("ZP_elec"))
    try:
        zp_elec = float(zp_elec)
    except (TypeError, ValueError):
        zp_elec = np.nan
    if np.isfinite(zp_elec):
        return data, zp_elec, "elec"

    zp_exp = source.get("photometric_zeropoint_exp", source.get("ZP_exp"))
    try:
        zp_exp = float(zp_exp)
    except (TypeError, ValueError):
        zp_exp = np.nan
    if np.isfinite(zp_exp):
        return data, zp_exp, "exp"

    raise KeyError(
        "Neither a finite photometric_zeropoint_elec nor a finite "
        "photometric_zeropoint_exp is available."
    )


def calculate_apparent_magnitude(object_flux, exptime, header, zeropoint, mode):
    """Calculate apparent magnitude with the selected zeropoint convention."""
    if not np.isfinite(object_flux) or object_flux <= 0:
        return np.nan, np.nan, np.nan

    instrumental_mag = -2.5 * np.log10(object_flux)
    if mode == "elec":
        if "EGAINSAV" not in header:
            raise KeyError(
                "photometric_zeropoint_elec is being used, but EGAINSAV is "
                "missing from the science FITS header."
            )
        egainsav = float(header["EGAINSAV"])
        if not np.isfinite(egainsav) or egainsav <= 0:
            raise ValueError(f"Invalid EGAINSAV: {egainsav}")
        apparent_mag = (
            instrumental_mag
            + 2.5 * np.log10(exptime)
            - 2.5 * np.log10(egainsav)
            + zeropoint
        )
    elif mode == "exp":
        egainsav = np.nan
        apparent_mag = instrumental_mag + 2.5 * np.log10(exptime) + zeropoint
    else:
        raise ValueError(f"Unknown zeropoint mode: {mode}")
    return instrumental_mag, apparent_mag, egainsav

### SER reading and Bayer channels

In [ ]:
BAYER_PATTERNS = {8: "RGGB", 9: "GRBG", 10: "GBRG", 11: "BGGR"}


def bayer_phases(pattern):
    """Row/column offset of each colour phase; green phases are named G1 and G2."""
    pattern = pattern.upper()
    phases = {}
    green = 0
    for k, letter in enumerate(pattern):
        dy, dx = divmod(k, 2)
        if letter == "G":
            green += 1
            phases[f"G{green}"] = (dy, dx)
        else:
            phases[letter] = (dy, dx)
    return phases


def channel_offset(name, phases):
    """
    Offset [superpixel] added to a superpixel coordinate to address the same
    sky position in a phase sub-image (phase sample at original pixel
    2i + dx; superpixel centre at original 2i + 0.5).
    """
    if name in ("ALL", "G"):
        return 0.0, 0.0
    dy, dx = phases[name]
    return 0.25 - dx / 2.0, 0.25 - dy / 2.0


def read_capture():
    """Frames of the selected range (float32), per-frame start times (astropy Time), header info."""
    ser = serparser.Serfile(str(PATH_capture))
    header = ser.getHeader()
    count = int(ser.getLength())
    first = 1 if frame_start is None else int(frame_start)
    last = count if frame_end is None else int(frame_end)
    if first < 1 or last > count or first > last:
        raise ValueError(f"Invalid frame range {first}--{last} (SER has {count} frames).")
    ticks = ser.readTrailFromHeader()
    if ticks is None or len(ticks) != count:
        raise RuntimeError("The SER file has no usable per-frame timestamps.")
    frames = np.stack([ser.readFrameAtPos(i).astype(np.float32) for i in tqdm(range(first - 1, last), desc="Reading SER")])
    times = Time([windows_ticks_to_time(ticks[i]).unix for i in range(first - 1, last)], format="unix", scale="utc")
    pattern = BAYER_PATTERNS.get(int(header["ColorID"])) if bayer_pattern == "auto" else bayer_pattern
    if pattern is None:
        raise ValueError(f"SER ColorID {header['ColorID']} is not a Bayer pattern; set bayer_pattern manually.")
    settings = parse_sharpcap_settings(PATH_capture.with_suffix(".CameraSettings.txt"))
    info = {
        "frame_first": first, "frame_last": last, "n_frames": last - first + 1,
        "width": int(header["ImageWidth"]), "height": int(header["ImageHeight"]),
        "bit_depth": int(header["PixelDepthPerPlane"]), "bayer_pattern": pattern,
        "exposure": float(settings.get("exposure", ser_exposure_seconds)),
        "gain": settings.get("gain", ser_gain), "offset": settings.get("offset", ser_offset),
        "camera": settings.get("camera_name", ser_camera_name) or str(header.get("Instrument", "")).strip(),
        "swcreate": settings.get("swcreate", "SharpCap"),
    }
    return frames, times, info


def split_channels(frames, phases):
    """Sub-images of each Bayer phase plus G (mean of greens) and ALL (sum of phases)."""
    ny2, nx2 = frames.shape[-2] // 2, frames.shape[-1] // 2
    out = {name: frames[..., dy:2 * ny2:2, dx:2 * nx2:2] for name, (dy, dx) in phases.items()}
    greens = [out[k] for k in out if k.startswith("G")]
    out["G"] = sum(greens) / len(greens)
    out["ALL"] = sum(out[k] for k in phases)
    return out

### Registration and star subtraction

In [ ]:
def phase_correlation_shift(image, reference):
    """(dx, dy) that moves `reference` onto `image`, with parabolic sub-pixel refinement."""
    a = np.nan_to_num(image - np.nanmedian(image))
    b = np.nan_to_num(reference - np.nanmedian(reference))
    A, B = np.fft.rfft2(a), np.fft.rfft2(b)
    R = A * np.conj(B)
    R /= np.abs(R) + 1e-9
    c = np.fft.fftshift(np.fft.irfft2(R, s=a.shape))
    py, px = np.unravel_index(int(np.argmax(c)), c.shape)
    cy, cx = a.shape[0] // 2, a.shape[1] // 2

    def refine(m, z, p):
        d = m - 2 * z + p
        return 0.0 if d == 0 else 0.5 * (m - p) / d

    sx = refine(c[py, px - 1], c[py, px], c[py, (px + 1) % c.shape[1]]) if 0 < px < c.shape[1] - 1 else 0.0
    sy = refine(c[py - 1, px], c[py, px], c[(py + 1) % c.shape[0], px]) if 0 < py < c.shape[0] - 1 else 0.0
    return float(px - cx + sx), float(py - cy + sy)


def measure_registration(all_frames, reference_index):
    shifts = np.array([phase_correlation_shift(f, all_frames[reference_index]) for f in tqdm(all_frames, desc="Measuring star drift")])
    print(f"Star drift: x {shifts[:, 0].min():+.2f}..{shifts[:, 0].max():+.2f}, y {shifts[:, 1].min():+.2f}..{shifts[:, 1].max():+.2f} superpixel")
    return shifts


def register(stack, shifts):
    """Shift every frame onto the reference grid (bilinear, NaN outside)."""
    return np.stack([ndimage.shift(stack[i], (-shifts[i, 1], -shifts[i, 0]), order=1, mode="constant", cval=np.nan)
                     for i in range(len(stack))])


def star_mask_from(median_image, sigma, dilation):
    finite = np.isfinite(median_image)
    _, med, std = sigma_clipped_stats(median_image[finite], sigma=3.0, maxiters=5)
    mask = finite & (median_image > med + sigma * std)
    return ndimage.binary_dilation(mask, iterations=int(dilation))

### Target detection (moving object)

In [ ]:
def residual_detections(residual, star_mask, sigma, min_area, per_frame):
    """Brightest residual blobs per frame outside the star mask: rows (frame, x, y, flux, area)."""
    finite = np.isfinite(residual[len(residual) // 2])
    _, med, std = sigma_clipped_stats(residual[len(residual) // 2][finite], sigma=3.0, maxiters=5)
    rows = []
    for i in range(len(residual)):
        sm = ndimage.gaussian_filter(np.nan_to_num(residual[i], nan=0.0), 1.0)
        sm[star_mask] = 0.0
        labels, n = ndimage.label(sm > med + sigma * std)
        cands = []
        for k, sl in enumerate(ndimage.find_objects(labels), start=1):
            reg = labels[sl] == k
            if np.count_nonzero(reg) < min_area:
                continue
            w = np.where(reg, sm[sl] - med, 0.0)
            t = w.sum()
            yy, xx = np.indices(w.shape)
            cands.append((t, (w * xx).sum() / t + sl[1].start, (w * yy).sum() / t + sl[0].start, int(np.count_nonzero(reg))))
        cands.sort(key=lambda c: -c[0])
        rows += [(i, c[1], c[2], c[0], c[3]) for c in cands[:per_frame]]
    return np.array(rows, dtype=float).reshape(-1, 5), float(std)


def fit_track_ransac(dets, elapsed, speed_expected, tolerance, iterations, inlier_distance, seed=0):
    """Straight track x(t), y(t) through residual detections with a speed prior [superpixel/s]."""
    rng = np.random.default_rng(seed)
    t = elapsed[dets[:, 0].astype(int)]
    best = (0, None)
    for _ in range(int(iterations)):
        a, b = rng.choice(len(dets), 2, replace=False)
        dt = t[b] - t[a]
        if abs(dt) < 3.0:
            continue
        v = (dets[b, 1:3] - dets[a, 1:3]) / dt
        speed = float(np.hypot(*v))
        if not (tolerance[0] * speed_expected <= speed <= tolerance[1] * speed_expected):
            continue
        pred = dets[a, 1:3] + np.outer(t - t[a], v)
        inl = np.hypot(*(dets[:, 1:3] - pred).T) < inlier_distance
        n = len(set(dets[inl, 0].astype(int)))
        if n > best[0]:
            best = (n, inl)
    if best[1] is None:
        raise RuntimeError("No straight track with the expected rate was found; check the rate prior or give manual anchors.")
    inl = best[1]
    for _ in range(3):
        px = np.polyfit(t[inl], dets[inl, 1], 1)
        py = np.polyfit(t[inl], dets[inl, 2], 1)
        r = np.hypot(dets[:, 1] - np.polyval(px, t), dets[:, 2] - np.polyval(py, t))
        inl = r < inlier_distance
    return {
        "vx": float(px[0]), "vy": float(py[0]), "x0": float(px[1]), "y0": float(py[1]),
        "n_inlier_frames": int(len(set(dets[inl, 0].astype(int)))),
        "n_frames_with_detection": int(len(set(dets[:, 0].astype(int)))),
        "rms_superpixel": float(np.sqrt(np.mean(r[inl] ** 2))),
        "inlier_rows": np.nonzero(inl)[0].tolist(),
    }


def track_position(track, elapsed_seconds):
    return track["x0"] + track["vx"] * elapsed_seconds, track["y0"] + track["vy"] * elapsed_seconds


def recentre(image, x0, y0, half_size, max_shift):
    """Flux-weighted centroid of the brightest smoothed blob near (x0, y0); falls back to (x0, y0)."""
    ny, nx = image.shape
    xs, ys = int(round(x0)) - half_size, int(round(y0)) - half_size
    xs, ys = max(0, xs), max(0, ys)
    sub = np.nan_to_num(image[ys:ys + 2 * half_size + 1, xs:xs + 2 * half_size + 1], nan=0.0)
    if sub.size < 9:
        return float(x0), float(y0), "track_prediction"
    src = np.clip(sub - np.median(sub), 0, None)
    sm = ndimage.gaussian_filter(src, 1.0)
    if not np.any(sm > 0):
        return float(x0), float(y0), "track_prediction"
    py, px = np.unravel_index(int(np.argmax(sm)), sm.shape)
    w = 3
    win = src[max(0, py - w):py + w + 1, max(0, px - w):px + w + 1]
    tot = float(win.sum())
    if tot <= 0:
        return float(x0), float(y0), "track_prediction"
    yy, xx = np.indices(win.shape)
    xc = float((win * xx).sum() / tot) + max(0, px - w) + xs
    yc = float((win * yy).sum() / tot) + max(0, py - w) + ys
    if np.hypot(xc - x0, yc - y0) > max_shift:
        return float(x0), float(y0), "track_prediction"
    return xc, yc, "local_centroid"


def cutout(image, x, y, half):
    ny, nx = image.shape
    x0, y0 = max(0, int(round(x)) - half), max(0, int(round(y)) - half)
    return image[y0:min(ny, int(round(y)) + half + 1), x0:min(nx, int(round(x)) + half + 1)], x0, y0


def create_review_figure(residual_all, residual_channels, dets, track, elapsed, output_path, title, n_cut=8):
    res_sum = np.nansum(residual_all, axis=0)
    n = len(residual_all)
    px_, py_ = track_position(track, elapsed)
    inl = np.zeros(len(dets), bool)
    inl[track["inlier_rows"]] = True
    names = ["ALL"] + [k for k in residual_channels]
    fig = plt.figure(figsize=(2.2 * n_cut, 6 + 1.9 * len(names)))
    grid = fig.add_gridspec(1 + len(names), n_cut, height_ratios=[4] + [1] * len(names), hspace=0.35)
    ax = fig.add_subplot(grid[0, :])
    ax.imshow(res_sum, cmap="gray", origin="lower", norm=ImageNormalize(res_sum, interval=ZScaleInterval(), stretch=AsinhStretch()))
    ax.plot(dets[inl, 1], dets[inl, 2], ".", color="lime", ms=3, label=f"track detections ({track['n_inlier_frames']} frames)")
    ax.plot(dets[~inl, 1], dets[~inl, 2], "x", color="red", ms=3, alpha=0.5, label="other residual blobs")
    ax.plot(px_, py_, "-", color="cyan", lw=0.8, label="fitted track")
    ax.plot(px_[0], py_[0], "o", color="cyan", ms=6, mfc="none", label="first frame")
    ax.set_title(f"{title}\nsum of star-subtracted registered frames; rate {np.hypot(track['vx'], track['vy']):.2f} superpixel/s")
    ax.legend(fontsize=8)
    pick = np.linspace(0, n - 1, n_cut).round().astype(int)
    for row, name in enumerate(names, start=1):
        imgs = residual_all if name == "ALL" else residual_channels[name]
        for k, i in enumerate(pick):
            a = fig.add_subplot(grid[row, k])
            sub, x0, y0 = cutout(imgs[i], px_[i], py_[i], 12)
            a.imshow(sub, cmap="gray", origin="lower", norm=ImageNormalize(sub, interval=ZScaleInterval(), stretch=AsinhStretch()))
            a.plot(px_[i] - x0, py_[i] - y0, "+", color="lime", ms=10)
            a.set_title(f"{name} frame {i + 1}", fontsize=7)
            a.set_xticks([]); a.set_yticks([])
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=110, bbox_inches="tight")
    plt.close(fig)
    return output_path

### Astrometry and zeropoints

In [ ]:
def read_center_hint():
    if astrometry_center_hint is None:
        return None
    if astrometry_center_hint != "auto":
        return float(astrometry_center_hint[0]), float(astrometry_center_hint[1])
    path = PATH_capture.with_suffix(".CameraSettings.txt")
    if not path.is_file():
        return None
    text = path.read_text(encoding="utf-8", errors="replace")
    pattern = r"RA=(\d+):(\d+):([\d.]+),\s*Dec=([+-]?\d+):(\d+):([\d.]+)"
    match = re.search(r"Telescope=" + pattern, text) or re.search(pattern, text)
    if match is None:
        return None
    h, m, s, dd, dm, ds = match.groups()
    ra = 15.0 * (float(h) + float(m) / 60.0 + float(s) / 3600.0)
    sign = -1.0 if dd.strip().startswith("-") else 1.0
    dec = sign * (abs(float(dd)) + float(dm) / 60.0 + float(ds) / 3600.0)
    return ra, dec


def plate_solve_image(image, header, work_directory, solved_path, center_hint):
    """solve-field on a FITS image; returns (header with WCS, rdls path) or raises."""
    work_directory = Path(work_directory)
    recreate_directory(work_directory)
    source = work_directory / "astrometry_input.fits"
    fits.writeto(source, np.nan_to_num(image, nan=float(np.nanmedian(image))).astype(np.float32), header, overwrite=True, output_verify="silentfix")
    command = ["solve-field", str(source), "--fits-image", "--crpix-center", "--no-plots", "--overwrite", "--tag-all",
               "-D", str(work_directory), "-N", str(solved_path), "--cpulimit", str(astrometry_cpulimit_seconds),
               "--scale-units", "arcsecperpix", "--scale-low", str(astrometry_scale_low_arcsec_per_pixel),
               "--scale-high", str(astrometry_scale_high_arcsec_per_pixel)]
    for key, value in astrometry_extra_options.items():
        if value is True:
            command.append(key)
        elif value not in (False, None):
            command += [key, str(value)]
    if center_hint is not None:
        command += ["--ra", str(center_hint[0]), "--dec", str(center_hint[1]), "--radius", str(astrometry_search_radius_deg)]
    completed = subprocess.run(command, capture_output=True, text=True)
    if not Path(solved_path).exists():
        tail = "\n".join(completed.stdout.strip().splitlines()[-4:])
        raise RuntimeError(f"solve-field failed: {tail}")
    with fits.open(solved_path) as hdul:
        solved_header = hdul[0].header.copy()
    rdls = sorted(work_directory.glob("*.rdls"))
    return solved_header, (rdls[0] if rdls else None)


def reference_stars(wcs, shape, rdls_path):
    """Gaia DR3 stars (or RDLS stars) inside the frame with pixel positions."""
    ny, nx = shape
    centre = wcs.pixel_to_world((nx - 1) / 2.0, (ny - 1) / 2.0)
    scales = proj_plane_pixel_scales(wcs) * 3600.0
    radius = 0.5 * np.hypot(nx * scales[0], ny * scales[1]) * u.arcsec
    if zeropoint_catalog == "gaia":
        from astroquery.gaia import Gaia
        Gaia.ROW_LIMIT = int(gaia_max_rows)
        job = Gaia.cone_search_async(centre, radius=radius,
                                     columns=["source_id", "ra", "dec", "phot_g_mean_mag", "phot_bp_mean_mag", "phot_rp_mean_mag"])
        table = job.get_results()
        table = table[np.isfinite(np.asarray(table["phot_g_mean_mag"], float))]
        catalogue = "Gaia DR3"
    elif zeropoint_catalog == "index":
        if rdls_path is None:
            raise RuntimeError("No RDLS file from solve-field.")
        rdls = Table.read(rdls_path)
        table = Table({"ra": rdls["RA"], "dec": rdls["DEC"], "phot_g_mean_mag": rdls["mag"],
                       "phot_bp_mean_mag": rdls["mag"], "phot_rp_mean_mag": rdls["mag"]})
        catalogue = "astrometry.net index (single band)"
    else:
        raise ValueError("zeropoint_catalog must be 'gaia' or 'index'.")
    sky = SkyCoord(ra=np.asarray(table["ra"], float) * u.deg, dec=np.asarray(table["dec"], float) * u.deg)
    x, y = wcs.world_to_pixel(sky)
    x, y = np.asarray(x, float), np.asarray(y, float)
    margin = zp_annulus_r_out + 2
    inside = np.isfinite(x) & np.isfinite(y) & (x > margin) & (x < nx - margin) & (y > margin) & (y < ny - margin)
    table = table[inside]
    table["x_wcs"], table["y_wcs"] = x[inside], y[inside]
    print(f"{catalogue}: {len(table)} reference stars inside the frame")
    return table, catalogue


def measure_channel_zeropoint(image, exptime, stars, band, offset):
    """Zeropoint of one channel image against catalogue magnitudes `band`."""
    x = np.asarray(stars["x_refined"], float) + offset[0]
    y = np.asarray(stars["y_refined"], float) + offset[1]
    pos = np.column_stack([x, y])
    mask = ~np.isfinite(image)
    ap = CircularAperture(pos, r=zp_aperture_radius)
    an = CircularAnnulus(pos, r_in=zp_annulus_r_in, r_out=zp_annulus_r_out)
    stats = ApertureStats(image, an, mask=mask, sigma_clip=SigmaClip(sigma=3.0, maxiters=5))
    raw = np.asarray(aperture_photometry(image, ap, mask=mask, method="exact")["aperture_sum"], float)
    net = raw - np.asarray(stats.median, float) * ap.area
    peak = np.array([np.nanmax(cutout(image, px, py, 2)[0]) if cutout(image, px, py, 2)[0].size else np.nan for px, py in pos])
    cat = np.asarray(stars[band], float)
    colour = np.asarray(stars["phot_bp_mean_mag"], float) - np.asarray(stars["phot_rp_mean_mag"], float)
    ok = np.isfinite(net) & (net > 0) & np.isfinite(cat) & (cat > reference_mag_min) & (cat < reference_mag_max)
    zp_each = np.full(len(net), np.nan)
    zp_each[ok] = cat[ok] + 2.5 * np.log10(net[ok])
    sel = ok.copy()
    if np.count_nonzero(sel) < 3:
        raise RuntimeError(f"Fewer than three reference stars for band {band}.")
    zp0 = np.nanmedian(zp_each[sel])
    sel &= np.abs(zp_each - zp0) < 0.5
    zp = float(np.nanmedian(zp_each[sel]))
    scatter = float(np.nanstd(zp_each[sel]))
    good_colour = sel & np.isfinite(colour)
    colour_term = float(np.polyfit(colour[good_colour], zp_each[good_colour] - zp, 1)[0]) if np.count_nonzero(good_colour) >= 5 else None
    return {
        "reference_band": band, "n_stars": int(np.count_nonzero(sel)),
        "photometric_zeropoint": zp, "photometric_zeropoint_exp": zp - 2.5 * np.log10(exptime),
        "zeropoint_scatter": scatter, "colour_term_per_bp_rp": colour_term,
        "stars": {"net_flux": net.tolist(), "zeropoint": zp_each.tolist(), "used": sel.tolist(), "peak": peak.tolist()},
    }

### Output

In [ ]:
def json_value(value):
    if np.ma.is_masked(value):
        return None
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    return value


def sanitize_json(value):
    if isinstance(value, dict):
        return {str(k): sanitize_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [sanitize_json(v) for v in value]
    return json_value(value)


def export_channel_json(table, output_path, channel, capture_info, zeropoint, target_info, elapsed_origin):
    """KUPT-lightcurveJSON version 2 for one channel."""
    valid = [row for row in table if str(row["time_start_utc"]).strip()]
    if not valid:
        raise ValueError("No valid timestamp.")
    start = Time(str(valid[0]["time_start_utc"]), scale="utc")
    end = Time(str(valid[-1]["time_start_utc"]), scale="utc") + TimeDelta(float(valid[-1]["exptime_seconds"]), format="sec")
    data_rows = []
    for row in table:
        data_rows.append({
            "id": int(row["frame_index"]),
            "time_start_utc": str(row["time_start_utc"]) or None,
            "time_mid_utc": str(row["time_mid_utc"]) or None,
            "elapsed_seconds": json_value(row["elapsed_seconds"]),
            "x_centroid": json_value(row["x_centroid"]),
            "y_centroid": json_value(row["y_centroid"]),
            "aperture_sum": json_value(row["aperture_sum_adu"]),
            "bkg_med_pix": json_value(row["background_median_adu_per_pixel"]),
            "object_flux": json_value(row["object_flux_adu"]),
            "mag_in": json_value(row["instrumental_mag"]),
            "mag_app": json_value(row["apparent_mag"]),
            "status": str(row["status"]),
            "comment": {"general": [f"Source: {PATH_capture.name} frame {int(row['ser_frame'])}",
                                    f"Position method: {row['position_method']}", f"Channel: {channel}"]},
        })
    output = {
        "header": {
            "object": {"objectName": object_name, "intlDES": intl_des, "noradID": None if norad_id is None else str(norad_id)},
            "startDateTime": start.isot, "endDateTime": end.isot, "length": len(table),
            "captureSettings": {
                "Xnaxis": capture_info["width"] // 2, "Ynaxis": capture_info["height"] // 2,
                "Xbinning": 2, "Ybinning": 2, "exposure": capture_info["exposure"],
                "gain": json_value(capture_info.get("gain")), "offset": json_value(capture_info.get("offset")),
                "camName": capture_info.get("camera"), "swcreate": capture_info.get("swcreate"),
                "bayerPattern": capture_info["bayer_pattern"], "channel": channel,
                "note": "superpixel grid (half resolution); ALL = sum of the four Bayer phases, G = mean of the two green phases",
            },
            "photParams": {
                "ZP": zeropoint["photometric_zeropoint"], "ZP_exp": zeropoint["photometric_zeropoint_exp"],
                "ZP_scatter": zeropoint["zeropoint_scatter"], "selected_mode": "exp",
                "reference_catalogue": zeropoint.get("catalogue"), "reference_band": zeropoint["reference_band"],
                "colour_term_per_bp_rp": zeropoint["colour_term_per_bp_rp"], "n_reference_stars": zeropoint["n_stars"],
            },
            "observer": {"observerName": observer_name, "observatory": observatory},
            "editor": {"appName": editor_app_name, "version": editor_version},
            "measureSettings": {
                "measureFieldShape": "CIRCLE",
                "measureFieldSize": {"aperture_radius_pixel": float(aperture_radius), "annulus_r_in_pixel": float(annulus_r_in),
                                     "annulus_r_out_pixel": float(annulus_r_out), "unit": "superpixel"},
                "targetPosition": sanitize_json(target_info),
            },
        },
        "data": data_rows,
        "createdBy": {"appName": creator_app_name, "version": creator_version,
                      "built": int(datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S"))},
        "dataType": "KUPT-lightcurveJSON", "version": 2,
        "comment": {"general": ["GEO sidereal-tracking photometry with a colour (Bayer) sensor: registered frames, "
                                "median field-star subtraction, automatic track detection, Gaia-based zeropoint per channel."],
                    "TLE": []},
    }
    output["header"]["captureSettings"] = {k: v for k, v in output["header"]["captureSettings"].items() if v is not None}
    output["header"]["photParams"] = {k: v for k, v in output["header"]["photParams"].items() if v is not None}
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(sanitize_json(output), file, ensure_ascii=False, indent=2, allow_nan=False)
    return output

### Stage 1: registration, star subtraction, track detection, review

In [ ]:
def capture_base_name():
    safe = re.sub(r"[^A-Za-z0-9._-]+", "_", object_name).strip("_")
    return f"{safe}_{PATH_capture.stem}"


def stage1_state_path():
    return PATH_work / f"stage1_color_state_{capture_base_name()}.json"


def stage1_detect():
    PATH_output.mkdir(parents=True, exist_ok=True)
    PATH_work.mkdir(parents=True, exist_ok=True)
    frames, times, info = read_capture()
    phases = bayer_phases(info["bayer_pattern"])
    chans = split_channels(frames, phases)
    elapsed = (times - times[0]).to_value("sec")
    print(f"{info['n_frames']} frames, {info['width']}x{info['height']} {info['bayer_pattern']} {info['bit_depth']}-bit, "
          f"exposure {info['exposure']} s, superpixel grid {chans['ALL'].shape[2]}x{chans['ALL'].shape[1]}")

    ref = (info["n_frames"] // 2) if registration_reference_frame is None else int(registration_reference_frame) - 1
    shifts = measure_registration(chans["ALL"], ref)
    all_reg = register(chans["ALL"], shifts)
    median_all = np.nanmedian(all_reg, axis=0)
    residual_all = all_reg - median_all
    fits.writeto(PATH_work / f"median_star_ALL_{capture_base_name()}.fits", median_all.astype(np.float32), overwrite=True)

    # pixel scale for the rate prior: solve one frame unless given
    scale = superpixel_scale_arcsec
    solved_header = None
    try:
        hdr = fits.Header()
        hdr["EXPTIME"] = info["exposure"]
        solved_header, _ = plate_solve_image(all_reg[ref], hdr, PATH_work / "astrometry_stage1", PATH_work / f"solved_stage1_{capture_base_name()}.fits", read_center_hint())
        scale = float(np.sqrt(np.abs(np.linalg.det(WCS(solved_header).pixel_scale_matrix))) * 3600.0)
        print(f"Reference frame solved: {scale:.4f} arcsec/superpixel")
    except Exception as error:
        print(f"Stage-1 plate solve failed ({error}); using superpixel_scale_arcsec = {scale}")
        if scale is None:
            raise
    speed_expected = expected_rate_arcsec_per_s / scale

    mask = star_mask_from(median_all, star_mask_sigma, star_mask_dilation)
    dets, noise = residual_detections(residual_all, mask, detection_sigma, detection_min_area, detections_per_frame)
    print(f"Residual blobs: {len(dets)} in {len(set(dets[:, 0].astype(int)))} frames (noise {noise:.1f} ADU)")

    if manual_anchor_1 is not None and manual_anchor_2 is not None:
        (f1, x1, y1), (f2, x2, y2) = manual_anchor_1, manual_anchor_2
        t1, t2 = elapsed[int(f1) - 1], elapsed[int(f2) - 1]
        track = {"vx": (x2 - x1) / (t2 - t1), "vy": (y2 - y1) / (t2 - t1), "x0": x1 - (x2 - x1) / (t2 - t1) * t1,
                 "y0": y1 - (y2 - y1) / (t2 - t1) * t1, "n_inlier_frames": 0, "n_frames_with_detection": 0,
                 "rms_superpixel": None, "inlier_rows": [], "method": "manual anchors"}
    else:
        track = fit_track_ransac(dets, elapsed, speed_expected, rate_tolerance, ransac_iterations, ransac_inlier_distance)
        track["method"] = "RANSAC on residual detections with GEO rate prior"
    rate = np.hypot(track["vx"], track["vy"]) * scale
    print(f"Track: v = ({track['vx']:.3f}, {track['vy']:.3f}) superpixel/s = {rate:.2f} arcsec/s, "
          f"{track['n_inlier_frames']} inlier frames, rms {track['rms_superpixel']}")
    px_, py_ = track_position(track, elapsed)
    print(f"Object: frame 1 ({px_[0]:.1f}, {py_[0]:.1f}) -> frame {info['n_frames']} ({px_[-1]:.1f}, {py_[-1]:.1f}) [registered superpixel grid]")

    residual_channels = {}
    for name in [c for c in channels if c != "ALL"]:
        reg = register(chans[name], shifts)
        residual_channels[name] = reg - np.nanmedian(reg, axis=0)
    review = create_review_figure(residual_all, residual_channels, dets, track, elapsed,
                                  PATH_output / "review" / f"{capture_base_name()}_track_review.png",
                                  f"{object_name} ({intl_des})  {PATH_capture.name} frames {info['frame_first']}-{info['frame_last']}")
    state = {
        "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "capture": str(PATH_capture), "info": info, "reference_index": int(ref),
        "shifts": shifts.tolist(), "superpixel_scale_arcsec": scale, "track": track,
        "detections": dets.tolist(), "review_figure": str(review),
        "solved_header": None if solved_header is None else solved_header.tostring(sep="\n"),
    }
    with stage1_state_path().open("w", encoding="utf-8") as file:
        json.dump(sanitize_json(state), file, ensure_ascii=False, indent=2)
    print(f"Review figure: {review}")
    print(f"Stage-1 state : {stage1_state_path()}")
    return state


def load_stage1_state():
    path = stage1_state_path()
    if not path.is_file():
        raise FileNotFoundError(f"Stage-1 state not found: {path}. Run stage 1 first.")
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)

### Stage 2: astrometry, zeropoints per channel, photometry

In [ ]:
def stage2_measure(state):
    frames, times, info = read_capture()
    phases = bayer_phases(info["bayer_pattern"])
    chans = split_channels(frames, phases)
    elapsed = (times - times[0]).to_value("sec")
    shifts = np.asarray(state["shifts"], float)
    ref = int(state["reference_index"])
    track = state["track"]
    n = info["n_frames"]
    product_directory = PATH_output / "lightcurve"
    product_directory.mkdir(parents=True, exist_ok=True)

    # ---- registration, calibration (after registration/astrometry), star subtraction per channel
    registered = {name: register(chans[name], shifts) for name in channels}
    summed = {name: np.nansum(registered[name], axis=0) for name in channels}      # stars x n for astrometry / zeropoint
    valid_count = np.sum(np.isfinite(registered["ALL"]), axis=0)

    # ---- astrometry on the registered ALL sum (raw data)
    hdr = fits.Header()
    hdr["EXPTIME"] = (info["exposure"] * n, "Summed exposure [s]")
    hdr["DATE-OBS"] = times[0].isot
    solved_header, rdls_path = plate_solve_image(summed["ALL"], hdr, PATH_work / "astrometry_stage2",
                                                 PATH_work / f"solved_stage2_{capture_base_name()}.fits", read_center_hint())
    wcs = WCS(solved_header)
    scale = float(np.sqrt(np.abs(np.linalg.det(wcs.pixel_scale_matrix))) * 3600.0)
    centre = wcs.pixel_to_world((summed["ALL"].shape[1] - 1) / 2.0, (summed["ALL"].shape[0] - 1) / 2.0)
    print(f"Plate solve: {scale:.4f} arcsec/superpixel, centre RA {centre.ra.deg:.4f} Dec {centre.dec.deg:.4f}")

    # ---- dark / flat (optional) applied to the registered channel frames
    if PATH_dark is not None or PATH_flat is not None:
        master_flat_dark = create_master_dark(PATH_flat_dark)
        master_dark = create_master_dark(PATH_dark)
        master_flat = create_master_flat(PATH_flat, master_flat_dark)
        scale_dark = 1.0 if calibration_scale == "auto" else float(calibration_scale)
        dark_ch = split_channels(master_dark * scale_dark, phases) if master_dark is not None else None
        flat_ch = split_channels(master_flat, phases) if master_flat is not None else None
        for name in channels:
            if dark_ch is not None:
                registered[name] = registered[name] - register(np.repeat(dark_ch[name][None], 1, axis=0), np.zeros((1, 2)))[0]
            if flat_ch is not None:
                with np.errstate(divide="ignore", invalid="ignore"):
                    registered[name] = registered[name] / flat_ch[name]
        summed = {name: np.nansum(registered[name], axis=0) for name in channels}
        print("Dark/flat applied to the registered channel frames.")

    # ---- reference stars and zeropoint per channel (on the registered sums)
    stars, catalogue = reference_stars(wcs, summed["ALL"].shape, rdls_path)
    xr, yr = centroid_sources(np.nan_to_num(summed["ALL"]), np.asarray(stars["x_wcs"], float), np.asarray(stars["y_wcs"], float),
                              box_size=zp_centroid_box, centroid_func=centroid_2dg)
    good = np.isfinite(xr) & np.isfinite(yr) & (np.hypot(xr - stars["x_wcs"], yr - stars["y_wcs"]) < zp_max_centroid_shift)
    stars["x_refined"] = np.where(good, xr, stars["x_wcs"])
    stars["y_refined"] = np.where(good, yr, stars["y_wcs"])
    zeropoints = {}
    for name in channels:
        zp = measure_channel_zeropoint(summed[name], info["exposure"] * n, stars, reference_band[name], channel_offset(name, phases))
        zp["catalogue"] = catalogue
        zeropoints[name] = zp
        ct = zp["colour_term_per_bp_rp"]
        print(f"Zeropoint {name:3s} vs {zp['reference_band']:16s}: ZP_exp = {zp['photometric_zeropoint_exp']:.3f}, "
              f"scatter {zp['zeropoint_scatter']:.3f} mag, {zp['n_stars']} stars, colour term {ct if ct is None else round(ct, 3)}")
    magzero_path = PATH_output / "magzero" / f"magzero_{times[0].isot[:19].replace(':', '')}_{capture_base_name()}_color.json"
    magzero_path.parent.mkdir(parents=True, exist_ok=True)
    with magzero_path.open("w", encoding="utf-8") as file:
        json.dump(sanitize_json({
            "metadata": {"creator": "Kiyoaki Okudaira", "created_at": datetime.now(ZoneInfo("Asia/Tokyo")).isoformat(timespec="seconds"),
                         "software": {"name": "SatPhotometry MAGZERO (GEO StarTrack colour)", "platform": platform.platform()}},
            "phot_params": {k: v for k, v in zeropoints["ALL"].items() if k != "stars"},
            "channels": {name: {k: v for k, v in zp.items() if k != "stars"} for name, zp in zeropoints.items()},
            "reference_stars": {"columns": ["ra", "dec", "phot_g_mean_mag", "phot_bp_mean_mag", "phot_rp_mean_mag", "x_refined", "y_refined"],
                                "data": [[json_value(stars[c][i]) for c in ["ra", "dec", "phot_g_mean_mag", "phot_bp_mean_mag", "phot_rp_mean_mag", "x_refined", "y_refined"]] for i in range(len(stars))],
                                "per_channel": {name: zp["stars"] for name, zp in zeropoints.items()}},
            "observation": {"capture": str(PATH_capture), "frames": [info["frame_first"], info["frame_last"]], "exposure": info["exposure"],
                            "gain": info.get("gain"), "camera": info.get("camera"), "bayer_pattern": info["bayer_pattern"],
                            "superpixel_scale_arcsec": scale, "field_centre_deg": [centre.ra.deg, centre.dec.deg]},
        }), file, ensure_ascii=False, indent=2)
    print(f"Saved MAGZERO JSON: {magzero_path}")

    # ---- star subtraction
    residual = {name: registered[name] - np.nanmedian(registered[name], axis=0) for name in channels}

    # ---- photometry per frame
    px_, py_ = track_position(track, elapsed)
    results = {name: [] for name in channels}
    n_sat = {name: 0 for name in channels}
    for i in tqdm(range(n), desc="Measuring"):
        x_obj, y_obj, method = recentre(residual["ALL"][i], px_[i], py_[i], recentre_half_size, recentre_max_shift)
        t_start = times[i]
        t_mid = t_start + TimeDelta(info["exposure"] / 2.0, format="sec")
        for name in channels:
            ox, oy = channel_offset(name, phases)
            image = residual[name][i]
            raw_img = registered[name][i]
            status = "Success"
            try:
                phot = measure_object(image, x_obj + ox, y_obj + oy, aperture_radius, annulus_r_in, annulus_r_out)
                peak_sub, _, _ = cutout(raw_img, x_obj + ox, y_obj + oy, int(np.ceil(aperture_radius)))
                peak = float(np.nanmax(peak_sub)) if peak_sub.size else np.nan
                phase_count = 4 if name == "ALL" else 1
                saturated = np.isfinite(peak) and peak >= saturation_level * (phase_count if name == "ALL" else 1) * 0.98
                zp = zeropoints[name]["photometric_zeropoint_exp"]
                flux = phot["object_flux"]
                if np.isfinite(flux) and flux > 0:
                    mag_in = -2.5 * np.log10(flux)
                    mag_app = mag_in + 2.5 * np.log10(info["exposure"]) + zp
                else:
                    mag_in = mag_app = np.nan
                    status = "Non-positive flux"
                if saturated and status == "Success":
                    status = "Saturated"
                results[name].append({
                    "frame_index": i, "ser_frame": info["frame_first"] + i, "channel": name,
                    "time_start_utc": t_start.isot, "time_mid_utc": t_mid.isot,
                    "elapsed_seconds": float(elapsed[i]), "exptime_seconds": info["exposure"],
                    "x_centroid": x_obj + ox, "y_centroid": y_obj + oy, "x_track": float(px_[i]), "y_track": float(py_[i]),
                    "position_method": method,
                    "aperture_sum_adu": phot["aperture_sum"], "background_median_adu_per_pixel": phot["background_median"],
                    "object_flux_adu": flux, "peak_adu": peak, "instrumental_mag": mag_in, "apparent_mag": mag_app,
                    "status": status,
                })
                if status == "Saturated":
                    n_sat[name] += 1
            except Exception as error:
                results[name].append({
                    "frame_index": i, "ser_frame": info["frame_first"] + i, "channel": name,
                    "time_start_utc": "", "time_mid_utc": "", "elapsed_seconds": np.nan, "exptime_seconds": info["exposure"],
                    "x_centroid": np.nan, "y_centroid": np.nan, "x_track": float(px_[i]), "y_track": float(py_[i]),
                    "position_method": method, "aperture_sum_adu": np.nan, "background_median_adu_per_pixel": np.nan,
                    "object_flux_adu": np.nan, "peak_adu": np.nan, "instrumental_mag": np.nan, "apparent_mag": np.nan,
                    "status": f"Error: {error}",
                })
    for name in channels:
        if n_sat[name]:
            print(f"Warning: {name}: {n_sat[name]} frame(s) saturated")

    # ---- outputs
    target_info = {"method": track.get("method"), "track": {k: v for k, v in track.items() if k != "inlier_rows"},
                   "superpixel_scale_arcsec": scale, "approved_by_operator": bool(require_confirmation and target_approved)}
    start = times[0]
    end = times[-1] + TimeDelta(info["exposure"], format="sec")
    safe_object = re.sub(r"[^A-Za-z0-9._-]+", "_", object_name).strip("_")
    safe_intl = re.sub(r"[^A-Za-z0-9._-]+", "_", str(intl_des)).strip("_")
    base = f"lightcurve_{safe_object}_{safe_intl}_{start.datetime.strftime('%Y-%m-%d_%H%M%S')}_{end.datetime.strftime('%H%M%S')}"
    tables = {}
    for name in channels:
        table = Table(rows=results[name])
        tables[name] = table
        table.write(product_directory / f"{base}_{name}.csv", format="ascii.csv", overwrite=True)
        export_channel_json(table, product_directory / f"{base}_{name}.json", name, info, zeropoints[name], target_info, times[0])
        print(f"Saved {name}: {product_directory / (base + '_' + name + '.csv')}")

    # combined table with colour indices
    combined = Table()
    combined["frame_index"] = tables["ALL"]["frame_index"]
    combined["ser_frame"] = tables["ALL"]["ser_frame"]
    combined["time_mid_utc"] = tables["ALL"]["time_mid_utc"]
    combined["elapsed_seconds"] = tables["ALL"]["elapsed_seconds"]
    combined["x_centroid"] = tables["ALL"]["x_centroid"]
    combined["y_centroid"] = tables["ALL"]["y_centroid"]
    for name in channels:
        combined[f"flux_{name}"] = tables[name]["object_flux_adu"]
        combined[f"mag_{name}"] = tables[name]["apparent_mag"]
        combined[f"status_{name}"] = tables[name]["status"]
    if "B" in channels and "R" in channels:
        combined["B-R"] = tables["B"]["apparent_mag"] - tables["R"]["apparent_mag"]
    if "B" in channels and "G" in channels:
        combined["B-G"] = tables["B"]["apparent_mag"] - tables["G"]["apparent_mag"]
    if "G" in channels and "R" in channels:
        combined["G-R"] = tables["G"]["apparent_mag"] - tables["R"]["apparent_mag"]
    combined_path = product_directory / f"{base}_color.csv"
    combined.write(combined_path, format="ascii.csv", overwrite=True)
    print(f"Saved combined table: {combined_path}")
    return tables, combined, base, zeropoints

### Run stage 1

In [ ]:
if run_stage in ("all", "1"):
    stage1_state = stage1_detect()
else:
    stage1_state = load_stage1_state()
    print(f"Loaded stage-1 state from {stage1_state_path()}")
display(Image(filename=stage1_state["review_figure"]))

### Review the detected track
The top panel shows the sum of the star-subtracted frames with the fitted track (cyan) and the frames in which
the object was detected (green). The rows below follow the track in the individual channels.

- Correct → set `target_approved = True` below and run the remaining cells.
- Wrong → give `manual_anchor_1` / `manual_anchor_2` (frame, x, y on the registered grid) or adjust the detection
  settings, and run stage 1 again.
- Unattended runs: `require_confirmation = False` or `SATPHOT_REQUIRE_CONFIRMATION=0`, or `SATPHOT_TARGET_APPROVED=1`.

In [ ]:
# target_approved = True

### Run stage 2

In [ ]:
if run_stage not in ("all", "2"):
    print("Stage 2 skipped (run_stage).")
elif require_confirmation and not target_approved:
    print("Stage 2 is waiting for the operator: review the figure, set target_approved = True, and run this cell again.")
else:
    lightcurve_tables, combined_table, output_base, zeropoint_results = stage2_measure(stage1_state)

### Result

In [ ]:
if "combined_table" in globals():
    colours = {"ALL": "black", "R": "tab:red", "G": "tab:green", "B": "tab:blue"}
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    t = np.asarray(combined_table["elapsed_seconds"], float)
    for name in channels:
        m = np.asarray(combined_table[f"mag_{name}"], float)
        ok = np.isfinite(m) & (np.asarray(combined_table[f"status_{name}"]) == "Success")
        axes[0].plot(t[ok], m[ok], ".-", color=colours.get(name, None), ms=3, lw=0.6, label=name)
    axes[0].invert_yaxis(); axes[0].set_ylabel("Apparent magnitude [mag]"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[0].set_title(f"{object_name} ({intl_des})  {PATH_capture.name}")
    for ci, col in (("B-R", "tab:purple"), ("G-R", "tab:orange"), ("B-G", "tab:cyan")):
        if ci in combined_table.colnames:
            v = np.asarray(combined_table[ci], float)
            axes[1].plot(t[np.isfinite(v)], v[np.isfinite(v)], ".", color=col, ms=3, label=ci)
    axes[1].set_ylabel("Colour index [mag]"); axes[1].set_xlabel("Elapsed time from first frame [s]"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(PATH_output / "lightcurve" / f"{output_base}_color.png", dpi=150)
    plt.show()